In [ ]:
# Create and validate our Spark session
from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .appName("projet-final") \
        .master("local[*]") \
        .getOrCreate()

print("Spark Version:", spark.version)

In [ ]:
# Test the connection to the HDFS

hdfs_root_path = "hdfs://namenode:9000/user/mti850/"

from pyspark.sql import functions as sf
from pyspark.sql.types import StringType

df_orders = spark.read.parquet(f"{hdfs_root_path}processed/olist_orders_dataset.parquet")
df_customers = spark.read.parquet(f"{hdfs_root_path}processed/olist_customers_dataset.parquet")

df_orders.show(1)
df_customers.show(1)

In [ ]:
# Calculate total_purchases for each client

df_purchases_by_client = (df_orders.join(df_customers, "customer_id")
                             .where(sf.col("order_status") == "delivered")
                             .groupBy(df_customers["customer_unique_id"])
                             .agg(sf.count(df_customers["customer_id"])
                                  .alias("total_purchases"))
                             .orderBy(sf.desc("total_purchases"))
                         )

df_purchases_by_client.show()

In [ ]:
# How many client have more than one purchase?

print(df_purchases_by_client.filter(sf.col("total_purchases") > 1).count())
print(df_purchases_by_client.filter(sf.col("total_purchases") <= 1).count())

Clients avec reachat : 2 801 \
Clients a achat unique : 90 557 \
Total de clients : 93 358

**Taux de reachat reel : 2 801 / 93 358 = 3.00 %**

#### Avoiding Data Leakage

If we give the model the statistics calculated over both years, the model "cheats" by reading the future. In production, a model never knows the future.

Your idea of ​​cutting time into blocks of time is called Time-Based Windowing in data science.

***“As soon as a customer receives and reviews their very first order, can we immediately predict whether they will repurchase on their own within 90 days, or whether they may never return?”***

#### Beware of Right Censoring:
Olist's data ends at the end of August 2018. If a customer places their first order on August 20, 2018, only 10 days have passed before the data set ends. We cannot say that he would not have redeemed within 90 days.

To be scientifically rigorous, we exclude the first orders placed in the last 90 days of the dataset (after May 2018).

In [ ]:
spark.stop()